# WikiArt Artist Classification — Deep Learning Project

**Goal:** Build a deep learning pipeline that can identify which artist painted a given artwork, using a subset of the WikiArt dataset (23 artists, ~13,000 paintings).

**Approach:**
1. **Explore** the dataset to understand its structure, class balance, and image characteristics
2. **Preprocess** images and build efficient data pipelines
3. **Train** multiple models (baseline CNN → transfer learning) and compare them
4. **Evaluate** thoroughly with metrics, confusion matrices, and error analysis

Each section below is self-contained — run the cells top to bottom.

---
## 1. Setup & Imports

We start by importing all project modules and setting random seeds for reproducibility.
Every experiment must be repeatable — if we re-run this notebook, we should get the same results.

In [ ]:
# Reload modules automatically when we edit the .py files
%load_ext autoreload
%autoreload 2

# Project modules
import config
from utils import set_seeds, plot_training_history
from data_exploration import (
    build_file_list,
    plot_class_distribution,
    analyse_image_dimensions,
    show_sample_images,
    compute_pixel_statistics,
)
from data_loader import prepare_all_datasets, encode_labels, split_dataset
from models import build_baseline_cnn, build_resnet50, build_efficientnet, unfreeze_base
from training import train_model, compile_model
from evaluation import evaluate_model

# Fix all random seeds for reproducibility
set_seeds()

print(f"Number of artist classes: {config.NUM_CLASSES}")
print(f"Artists: {', '.join(name.replace('_', ' ') for name in config.CLASS_NAMES)}")

---
## 2. Data Exploration

Before building any model, we need to deeply understand our data. This section answers:
- **How many paintings does each artist have?** (Are classes balanced or imbalanced?)
- **What do the images look like?** (Style, subject, colour palette)
- **What are the image dimensions?** (Do we lose information when resizing to 224×224?)
- **What are the pixel statistics?** (Should we use ImageNet normalisation or our own?)

These insights directly inform our modelling decisions.

### 2.1 Build file catalogue

Scan the `wikiart/` directory and create a flat list of all image paths and their corresponding artist labels.

In [ ]:
# Collect all image paths and their artist labels
file_paths, labels = build_file_list()
print(f"Total images found: {len(file_paths)}")

### 2.2 Class distribution

Check how many paintings each artist has. If classes are heavily imbalanced, the model might learn to always predict the majority class. We'll use this information to decide on class weighting during training.

In [ ]:
# Bar chart: number of paintings per artist
counts = plot_class_distribution(labels)

### 2.3 Sample paintings

Visual inspection of example paintings from each artist. This helps us build intuition about the visual differences between artists — some have distinctive colour palettes (e.g. Van Gogh), while others may share similar styles (e.g. Impressionists).

In [ ]:
# Show 3 example paintings per artist in a grid
show_sample_images(file_paths, labels, n_per_artist=3)

### 2.4 Image dimensions

Analyse the original image sizes. Since neural networks require fixed-size inputs, we need to resize all images to a common size (224×224). Understanding the original dimensions tells us how much spatial information is lost or distorted during resizing.

In [ ]:
# Scan a sample of images for width, height, and aspect ratio statistics
widths, heights = analyse_image_dimensions(file_paths, sample_size=500)

### 2.5 Pixel statistics

Compute the mean and standard deviation of pixel values per colour channel (Red, Green, Blue). We compare these to the ImageNet statistics to decide whether the standard ImageNet normalisation is appropriate for our painting dataset.

In [ ]:
# Compute per-channel (R, G, B) mean and std across the dataset
mean, std = compute_pixel_statistics(file_paths, sample_size=300)

---
## 3. Data Preprocessing & Pipeline

Now that we understand the data, we prepare it for training:
1. **Stratified split** into train (70%), validation (15%), and test (15%) — stratified means each split preserves the artist distribution
2. **Label encoding** — convert artist names to integer indices
3. **tf.data pipeline** — efficient loading, resizing, batching, and prefetching
4. **Data augmentation** — random flips, brightness, and contrast changes (training set only) to reduce overfitting

In [ ]:
# Split the dataset and build tf.data pipelines with augmentation
# This single call handles: stratified split → label encoding → tf.data pipeline
train_ds, val_ds, test_ds, test_paths, test_labels = prepare_all_datasets(file_paths, labels)

In [ ]:
# Quick sanity check: visualise one batch to confirm images and labels look correct
import matplotlib.pyplot as plt

images, batch_labels = next(iter(train_ds))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(images[i].numpy())
    artist = config.CLASS_NAMES[batch_labels[i].numpy()].replace("_", " ")
    ax.set_title(artist, fontsize=10)
    ax.axis("off")
fig.suptitle("Sample training batch (with augmentation applied)", fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. Model Training

We train three models in increasing order of complexity. This iterative approach lets us see exactly how much each technique contributes:

| # | Model | Strategy | Why |
|---|-------|----------|-----|
| 1 | **Baseline CNN** | Built from scratch | Lower-bound reference — how far can we get without pretrained knowledge? |
| 2 | **ResNet50** | Transfer learning (frozen base) | Leverage ImageNet features for painting classification |
| 3 | **EfficientNetB0** | Transfer learning (frozen base) | Compare a more parameter-efficient architecture against ResNet |

All models use the same data, augmentation, class weights, and callbacks for a fair comparison.

In [ ]:
# We need training labels as a numpy array for class-weight computation
from data_loader import split_dataset, encode_labels
(train_p, train_l), (val_p, val_l), (test_p, test_l) = split_dataset(file_paths, labels)
train_labels_encoded = encode_labels(train_l)

### 4.1 Baseline CNN

A simple 3-block convolutional network built from scratch. It has no pretrained knowledge, so it must learn everything from our ~9,000 training images alone. We expect moderate performance — this is our reference point.

In [ ]:
# Build and inspect the baseline CNN architecture
baseline_cnn = build_baseline_cnn()
baseline_cnn.summary()

In [ ]:
# Train the baseline CNN
baseline_history = train_model(
    baseline_cnn, train_ds, val_ds, train_labels_encoded,
    model_name="Baseline_CNN",
    epochs=config.EPOCHS,
    learning_rate=1e-3,  # higher LR for training from scratch
)

In [ ]:
# Plot training/validation accuracy and loss curves
plot_training_history(baseline_history, model_name="Baseline_CNN")

### 4.2 ResNet50 — Transfer Learning

ResNet50 was pre-trained on ImageNet (1.2M images, 1000 classes). Its convolutional layers already know how to detect edges, textures, shapes, and objects. We freeze these layers and only train a new classification head on top — effectively reusing the visual knowledge for our painting task.

In [ ]:
# Build ResNet50 with frozen backbone (feature extraction mode)
resnet_model = build_resnet50(freeze_base=True)
resnet_model.summary(show_trainable=True)

In [ ]:
# Train ResNet50 (frozen base — only the classification head learns)
resnet_history = train_model(
    resnet_model, train_ds, val_ds, train_labels_encoded,
    model_name="ResNet50_Transfer",
    epochs=config.EPOCHS,
    learning_rate=config.LEARNING_RATE,
)

In [ ]:
plot_training_history(resnet_history, model_name="ResNet50_Transfer")

### 4.3 ResNet50 — Fine-tuning

After the classification head has converged, we unfreeze the later layers of ResNet50 and continue training with a much lower learning rate. This lets the pretrained features adapt to the specific visual characteristics of paintings (brushstroke patterns, colour palettes) rather than generic ImageNet features.

In [ ]:
# Unfreeze the last ~30% of ResNet layers for fine-tuning
# We keep early layers frozen because they detect generic features
# (edges, colours) that are already well-suited for any vision task.
unfreeze_base(resnet_model, unfreeze_from_layer=140)

# Re-compile with a much lower learning rate to avoid destroying pretrained weights
resnet_ft_history = train_model(
    resnet_model, train_ds, val_ds, train_labels_encoded,
    model_name="ResNet50_FineTuned",
    epochs=15,  # fewer epochs — we're refining, not learning from scratch
    learning_rate=1e-5,
)

In [ ]:
plot_training_history(resnet_ft_history, model_name="ResNet50_FineTuned")

### 4.4 EfficientNetB0 — Transfer Learning

EfficientNet uses a compound scaling method that uniformly scales network width, depth, and resolution. It achieves better accuracy per parameter than ResNet, making it an interesting comparison. We follow the same two-phase approach: frozen base → fine-tuning.

In [ ]:
# Build EfficientNetB0 with frozen backbone
effnet_model = build_efficientnet(freeze_base=True)
effnet_model.summary(show_trainable=True)

In [ ]:
# Train EfficientNetB0 (frozen base)
effnet_history = train_model(
    effnet_model, train_ds, val_ds, train_labels_encoded,
    model_name="EfficientNetB0_Transfer",
    epochs=config.EPOCHS,
    learning_rate=config.LEARNING_RATE,
)

In [ ]:
plot_training_history(effnet_history, model_name="EfficientNetB0_Transfer")

### 4.5 EfficientNetB0 — Fine-tuning

Unfreeze the later layers and fine-tune with a low learning rate, just as we did for ResNet50.

In [ ]:
# Unfreeze the last ~30% of EfficientNet layers for fine-tuning
unfreeze_base(effnet_model, unfreeze_from_layer=200)

effnet_ft_history = train_model(
    effnet_model, train_ds, val_ds, train_labels_encoded,
    model_name="EfficientNetB0_FineTuned",
    epochs=15,
    learning_rate=1e-5,
)

In [ ]:
plot_training_history(effnet_ft_history, model_name="EfficientNetB0_FineTuned")

---
## 5. Evaluation & Comparison

We evaluate each model on the **test set** — data the models have never seen during training. This section provides:
- **Accuracy** — overall percentage of correct predictions
- **Classification report** — per-artist precision, recall, and F1-score
- **Confusion matrix** — which artists get confused with each other
- **Top-k accuracy** — how often the correct artist is in the model's top 3 or top 5 guesses
- **Misclassified examples** — visual inspection of the model's mistakes

### 5.1 Baseline CNN — Test Evaluation

In [ ]:
baseline_results = evaluate_model(
    baseline_cnn, test_ds, test_paths, test_labels,
    model_name="Baseline_CNN"
)

### 5.2 ResNet50 (Fine-tuned) — Test Evaluation

In [ ]:
resnet_results = evaluate_model(
    resnet_model, test_ds, test_paths, test_labels,
    model_name="ResNet50_FineTuned"
)

### 5.3 EfficientNetB0 (Fine-tuned) — Test Evaluation

In [ ]:
effnet_results = evaluate_model(
    effnet_model, test_ds, test_paths, test_labels,
    model_name="EfficientNetB0_FineTuned"
)

### 5.4 Model Comparison

Side-by-side comparison of all models to clearly see the impact of transfer learning and fine-tuning.

In [ ]:
import pandas as pd

# Collect all results into a comparison table
comparison = pd.DataFrame({
    "Model": ["Baseline CNN", "ResNet50 (fine-tuned)", "EfficientNetB0 (fine-tuned)"],
    "Test Accuracy": [
        baseline_results["accuracy"],
        resnet_results["accuracy"],
        effnet_results["accuracy"],
    ],
    "Top-3 Accuracy": [
        baseline_results.get("Top-3", None),
        resnet_results.get("Top-3", None),
        effnet_results.get("Top-3", None),
    ],
    "Top-5 Accuracy": [
        baseline_results.get("Top-5", None),
        resnet_results.get("Top-5", None),
        effnet_results.get("Top-5", None),
    ],
})

# Format as percentages for readability
for col in ["Test Accuracy", "Top-3 Accuracy", "Top-5 Accuracy"]:
    comparison[col] = comparison[col].apply(lambda x: f"{x:.2%}" if x else "N/A")

print("\n" + "="*60)
print("MODEL COMPARISON — TEST SET RESULTS")
print("="*60)
comparison

---
## 6. Summary & Next Steps

**Key findings to discuss in the report:**
- How much does transfer learning improve over training from scratch?
- Does fine-tuning the backbone give additional gains over a frozen base?
- Which artists are easiest/hardest to classify, and why?
- Which artists does the model confuse most, and does this align with art-historical relationships?
- How does class imbalance affect per-artist performance?

**Potential next steps:**
- Experiment with stronger augmentation (rotation, cutout, mixup)
- Try larger backbones (ResNet101, EfficientNetB3)
- Ensemble multiple models for improved accuracy
- Apply Grad-CAM to visualise which parts of the painting the model focuses on